# B2-020-language-transformers — Practice p09 — Solution

**Type:** constrained-coding · **Difficulty:** advanced · **Concepts:** language-transformer

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Solution

Add token and sinusoidal position vectors, call one causal block once, then apply a separate vocabulary head.

In [ ]:
import math
import torch
from torch import nn

ATOL = RTOL = 1e-6

class CausalBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = nn.MultiheadAttention(8, 2, batch_first=True, dropout=0.0)
        self.norm = nn.LayerNorm(8)
        self.calls = 0
    def forward(self, x, causal_mask):
        self.calls += 1
        y, _ = self.attention(x, x, x, attn_mask=causal_mask, need_weights=False)
        return self.norm(x + y)

def tiny_causal_lm_forward(token_ids, token_embedding, sinusoidal_table, block, vocab_head):
    length = token_ids.shape[1]
    hidden = token_embedding(token_ids) + sinusoidal_table[:length]
    causal_mask = torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)
    hidden = block(hidden, causal_mask)
    return vocab_head(hidden)

torch.manual_seed(20260812)
token_embedding = nn.Embedding(12, 8)
vocab_head = nn.Linear(8, 12)
positions = torch.arange(8, dtype=torch.float32).unsqueeze(1)
frequencies = torch.exp(torch.arange(0, 8, 2) * (-math.log(10000.0) / 8))
sinusoidal_table = torch.zeros(8, 8)
sinusoidal_table[:, 0::2] = torch.sin(positions * frequencies)
sinusoidal_table[:, 1::2] = torch.cos(positions * frequencies)
tokens = torch.tensor([[2,4,6,8,10,3,0], [2,5,7,8,11,3,0]], dtype=torch.int64)
block = CausalBlock()
logits = tiny_causal_lm_forward(tokens, token_embedding, sinusoidal_table, block, vocab_head)
mutated = tokens.clone(); mutated[:, 5] = torch.tensor([9, 4])
mutated_logits = tiny_causal_lm_forward(mutated, token_embedding, sinusoidal_table, block, vocab_head)

### Answer check

In [ ]:
assert logits.shape == (2, 7, 12) and logits.dtype == torch.float32
assert block.calls == 2
assert vocab_head.weight.data_ptr() != token_embedding.weight.data_ptr()
assert torch.allclose(logits[:, :5], mutated_logits[:, :5], atol=ATOL, rtol=RTOL)